In [ ]:
!pip -q install transformers
!pip -q install sentencepiece
!pip -q install accelerate
!pip -q install streamlit
!pip -q install pyngrok

In [ ]:
from transformers import pipeline

In [ ]:
sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

print("Model Loaded Successfully!")

In [ ]:
print(
    sentiment(
        "I am very happy with your service."
    )
)

In [ ]:
print(
    sentiment(
        "I am extremely disappointed."
    )
)

In [ ]:
def detect_sentiment(text):

    result = sentiment(text)

    return result[0]["label"], result[0]["score"]

In [ ]:
label, score = detect_sentiment(
    "Your service is amazing!"
)

print(label)

print(score)

In [ ]:
examples = [

    "I love this chatbot.",

    "Your support is terrible.",

    "Thank you very much.",

    "I hate waiting."

]

for sentence in examples:

    label, score = detect_sentiment(sentence)

    print(sentence)

    print(label)

    print(score)

    print("-"*40)

In [ ]:
def chatbot_response(message):

    label, score = detect_sentiment(message)

    if label == "POSITIVE":

        response = (
            "😊 I'm glad you're having a positive experience! "
            "How else can I assist you today?"
        )

    elif label == "NEGATIVE":

        response = (
            "😔 I'm sorry to hear that. "
            "Please tell me more about the issue so I can help."
        )

    else:

        response = (
            "🙂 Thank you for your message. "
            "How may I help you?"
        )

    return label, score, response

In [ ]:
label, score, response = chatbot_response(
    "I love your chatbot!"
)

print("Sentiment:", label)
print("Confidence:", score)
print("Bot:", response)

In [ ]:
label, score, response = chatbot_response(
    "Your service is terrible."
)

print("Sentiment:", label)
print("Confidence:", score)
print("Bot:", response)

In [ ]:
chat_history = []

In [ ]:
def customer_support_chatbot():

    print("="*60)
    print("CUSTOMER SUPPORT CHATBOT")
    print("="*60)

    while True:

        message = input("\nYou: ")

        if message.lower() == "exit":
            break

        label, score, response = chatbot_response(message)

        chat_history.append({
            "User": message,
            "Sentiment": label,
            "Confidence": score,
            "Bot": response
        })

        print("\nDetected Sentiment:", label)
        print("Confidence:", round(score,3))
        print("\nBot:", response)

In [ ]:
customer_support_chatbot()

In [ ]:
import pandas as pd

history_df = pd.DataFrame(chat_history)

history_df

In [ ]:
history_df.to_csv(
    "chat_history.csv",
    index=False
)

print("Chat History Saved!")

In [ ]:
positive = sum(
    1 for chat in chat_history
    if chat["Sentiment"] == "POSITIVE"
)

negative = sum(
    1 for chat in chat_history
    if chat["Sentiment"] == "NEGATIVE"
)

total = len(chat_history)

print("="*40)
print("CUSTOMER SATISFACTION REPORT")
print("="*40)

print("Total Conversations :", total)
print("Positive :", positive)
print("Negative :", negative)

if total > 0:

    satisfaction = positive / total * 100

    print("Customer Satisfaction:",
          round(satisfaction,2),
          "%")

In [ ]:
import matplotlib.pyplot as plt

counts = history_df["Sentiment"].value_counts()

plt.figure(figsize=(5,4))

counts.plot(kind="bar")

plt.title("Sentiment Distribution")

plt.xlabel("Sentiment")

plt.ylabel("Count")

plt.show()

In [ ]:
counts.plot(
    kind="pie",
    autopct="%1.1f%%",
    figsize=(5,5)
)

plt.ylabel("")
plt.title("Customer Sentiments")

plt.show()

In [ ]:
!pip install -q wordcloud

In [ ]:
from wordcloud import WordCloud

text = " ".join(history_df["User"])

wc = WordCloud(
    width=1000,
    height=500,
    background_color="white"
).generate(text)

plt.figure(figsize=(12,6))

plt.imshow(wc)

plt.axis("off")

plt.show()

In [ ]:
%%writefile app.py

import streamlit as st
from transformers import pipeline
import pandas as pd

# -----------------------------
# Load Sentiment Model
# -----------------------------

sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# -----------------------------
# Detect Sentiment
# -----------------------------

def detect_sentiment(text):

    result = sentiment(text)

    return result[0]["label"], result[0]["score"]

# -----------------------------
# Generate Response
# -----------------------------

def chatbot_response(message):

    label, score = detect_sentiment(message)

    if label == "POSITIVE":

        response = (
            "😊 Thank you! We're happy you're satisfied."
        )

    elif label == "NEGATIVE":

        response = (
            "😔 We're sorry you're facing this issue. "
            "Please tell us more so we can help."
        )

    else:

        response = (
            "🙂 Thank you for contacting us."
        )

    return label, score, response

# -----------------------------
# Streamlit UI
# -----------------------------

st.set_page_config(
    page_title="Sentiment Chatbot"
)

st.title("😊 Sentiment Analysis Chatbot")

message = st.text_area(
    "Enter your message"
)

if st.button("Analyze"):

    label, score, response = chatbot_response(message)

    st.subheader("Detected Sentiment")

    st.write(label)

    st.write("Confidence:", round(score,3))

    st.subheader("Bot Response")

    st.success(response)

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

In [ ]:
history_df.to_csv(
    "chat_history.csv",
    index=False
)

print("Chat history saved.")

In [ ]:
from google.colab import files

files.download("chat_history.csv")